# 국회도서관 API 활용: 논문 검색·통계·엑셀 저장 실습

**과제 목표:** `requests`로 국회도서관 자료검색 XML 수집 → HTTP/응답 구조 확인 → `pandas` 통계 → Excel 저장 → `assert` 검증.

API: [공공데이터포털](https://www.data.go.kr/) — **국회 국회도서관_자료검색 서비스**  
통합검색 주소: `https://apis.data.go.kr/9720000/searchservice/basic`

> 문서에는 HTTP URL이 예시로 제시되지만 실습은 인증키 보안을 위해 HTTPS로 요청합니다. HTTPS 미지원이나 접근 오류는 기관에 확인하세요. 키는 `.env`의 `DATA_GO_KR_API_KEY`에서만 읽습니다. 이 노트북은 실행 결과가 없는 제출용 템플릿으로, 실제 제출 전 각 셀을 실행해야 합니다.


## 셀 1 — 라이브러리와 API 인증키 설정

In [ ]:
# 최초 1회 필요한 경우 주석 해제:
# %pip install requests pandas openpyxl python-dotenv openai PyMuPDF reportlab

import os
import requests
import pandas as pd
from xml.etree import ElementTree as ET
from dotenv import load_dotenv
from paperlab import API_BASE_URL, parse_search_xml, calculate_trends, normalize_key
from reporting import create_excel

load_dotenv()
API_KEY = os.getenv("DATA_GO_KR_API_KEY")
print("API 키 설정 여부:", bool(API_KEY))  # 실제 키 값 출력 금지


## 셀 2 — 별도 설정 검증 셀

In [ ]:
assert API_KEY is not None and API_KEY.strip() != "", "인증키 미설정"
assert API_BASE_URL.endswith("/searchservice/basic"), "호출 경로가 올바르지 않습니다."
assert API_BASE_URL.startswith("https://")
print("기본 설정 검증 통과")


## 셀 3 — 검색어 입력 및 실제 HTTP 요청

In [ ]:
# 검색어 변경 가능: 예) 데이터베이스 포렌식
keyword = input("논문 주제 입력 (기본: 데이터베이스 포렌식): ").strip() or "데이터베이스 포렌식"
params = {
    "serviceKey": normalize_key(API_KEY),
    "pageno": 1,
    "displaylines": 10,
    "search": f"전체,{keyword}",
}

# 인증키는 URL에 포함되어 로그에 남을 수 있으므로 response.url 은 출력하지 않습니다.
response = requests.get(API_BASE_URL, params=params, timeout=(6, 25))
print("HTTP 상태 코드:", response.status_code)
print("응답 형식:", response.headers.get("Content-Type"))
print("XML 원문 첫 500자 (인증키가 포함되지 않은 응답만):")
print(response.text[:500])


## 셀 4 — XML 응답 분석 및 DataFrame 변환

In [ ]:
# 공공데이터포털의 오류 XML도 식별하는 함수 사용
api_total, records = parse_search_xml(response.content)
df = pd.DataFrame(records)

print("API 전체 결과 수:", api_total)
print("이번 페이지의 수집 수:", len(df))
print("XML 태그 예: record / item / name / value")

if not df.empty:
    display(df.head(10))
else:
    print("수집된 자료가 없습니다. 검색어를 변경해 보세요.")


## 셀 5 — API 응답 검증 셀

In [ ]:
assert response.status_code == 200, "HTTP 요청이 실패했습니다."
root = ET.fromstring(response.content)
assert root.findtext(".//resultCode") == "00", "API 내부 결과 코드 확인 필요"
assert isinstance(df, pd.DataFrame), "DataFrame 생성 실패"
assert len(df) > 0, "검색 결과가 없습니다. 다른 검색어로 다시 실행하세요."
assert "controlno" in df.columns, "관리번호 필드가 없습니다."
print("실제 API 검색 및 DataFrame 검증 통과")


## 셀 6 — pandas 데이터 분석 (결측·중복·연도별·유형별)

In [ ]:
print("결측치 건수:")
print(df.replace(r"^\s*$", pd.NA, regex=True).isna().sum())

print("
중복 자료 수:", df.duplicated(subset=["controlno"]).sum())

year_df, type_df = calculate_trends(df)
print("
발행연도별 건수")
display(year_df)
print("
자료 유형별 건수")
display(type_df)

if not year_df.empty:
    display(year_df.set_index("발행연도").plot.bar(y="건수", title="수집 자료 발행연도별 건수").figure)


## 셀 7 — 통계 검증 셀

In [ ]:
assert len(df) <= 10, "페이지당 10건 요청과 결과 수를 확인하세요."
assert int(type_df["건수"].sum()) == len(df), "유형별 건수 합계 불일치"
assert df["controlno"].duplicated().sum() == 0, "관리번호 중복이 있습니다."
assert set(year_df.columns) == {"발행연도", "건수"}
print("통계 검증 통과")


## 셀 8 — 제출용 Excel 파일 저장

In [ ]:
excel_bytes = create_excel(df, keyword)
file_name = "논문_API_실습_분석.xlsx"
with open(file_name, "wb") as file:
    file.write(excel_bytes)
print("Excel 저장 완료:", file_name)


## 셀 9 — Excel 저장 결과 검증 셀

In [ ]:
from openpyxl import load_workbook

assert os.path.isfile(file_name), "Excel 파일이 없습니다."
wb = load_workbook(file_name, read_only=True)
assert "검색결과" in wb.sheetnames
assert "연도별통계" in wb.sheetnames
assert "유형별통계" in wb.sheetnames
assert "참고문헌" in wb.sheetnames
assert wb["검색결과"].max_row == len(df) + 1
print("Excel 검증 통과:", wb.sheetnames)
wb.close()


## 제출 확인

- [ ] 셀 2, 5, 7, 9의 `assert` 검증 통과
- [ ] 키가 저장되지 않은 노트북 `.ipynb` 파일 제출
- [ ] API 수집·통계·출처가 들어간 `논문_API_실습_분석.xlsx` 제출
- [ ] API 이름 및 출처 URL 기재

**중요:** 결과의 '자료명/저자사항'은 제목과 저자가 결합된 서지 필드일 수 있습니다. 해당 API는 초록·논문 본문 제공을 보장하지 않습니다. AI 6항목 요약은 Streamlit 앱에서 논문 초록이나 PDF를 제공한 뒤 실행합니다.
